# Stage 2 — Data Cleaning & Wrangling
## Glassdoor Data-Science Jobs · Tier A pipeline

Stage 1 established that the file is materially incomplete behind a clean `isnull()`, and that
`Salary Estimate` indexes the scrape rather than the job. This notebook turns that raw file into two
analysis bases, and every decision it takes is logged.

**The three decisions that matter, stated before the code:**

1. **Sentinels become `NaN` and are counted — nothing is imputed.** Missingness here is MNAR by
   mechanism: Glassdoor omits company metadata for smaller and newer employers. Imputing it would
   fabricate exactly the segment the market-structure story is about. Models impute *inside* the
   pipeline, on training folds only.
2. **The artifact is materialised as a column.** `batch_id` makes the scrape-batch structure explicit
   so downstream analysis can group on it and test against it, instead of re-deriving it each time.
3. **Duplicates are treated at three levels, not one** — and the middle level is *deliberately
   preserved*, because those rows are the evidence for the report's central finding.

**Governs:** `DOCS/STRUCTURE.md` §2 · `IMPLEMENTATION_PLAN.md` §4

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_CSV = ROOT / "Uncleaned_DS_jobs.csv" / "Uncleaned_DS_jobs.csv"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

DTYPES = {
    "index": "int64", "Job Title": "string", "Salary Estimate": "string",
    "Job Description": "string", "Rating": "float64", "Company Name": "string",
    "Location": "string", "Headquarters": "string", "Size": "string", "Founded": "int64",
    "Type of ownership": "string", "Industry": "string", "Sector": "string",
    "Revenue": "string", "Competitors": "string",
}
raw = pd.read_csv(RAW_CSV, dtype=DTYPES, keep_default_na=False, na_values=[])
df = raw.copy()

# Every cleaning action appends one row here.
LOG: list[dict] = []
def log(action, column, rows_affected, why):
    LOG.append({"action": action, "column": column,
                "rows_affected": int(rows_affected), "why": why})

print(f"loaded {len(df):,} x {df.shape[1]}")

loaded 672 x 15


## 2.1 Sentinel missingness — convert and count, never impute

Two sentinel systems: `-1` across eight columns, and `"Unknown"` / `"Unknown / Non-Applicable"` in
`Size`, `Type of ownership` and `Revenue`. `Revenue` carries **both**, which is why its true
missingness (35.7%) is nearly nine times the 4.0% a `-1` count alone would report.

In [2]:
SENTINEL_COLS = ["Headquarters", "Size", "Type of ownership", "Industry", "Sector",
                 "Revenue", "Competitors"]

before = {c: int((df[c].str.strip() == "-1").sum()) for c in SENTINEL_COLS}
for c in SENTINEL_COLS:
    df[c] = df[c].replace("-1", pd.NA)

# "Unknown" family - a second, independent sentinel system
unknown_mask = {c: df[c].str.contains("Unknown", na=False) for c in ["Size", "Type of ownership", "Revenue"]}
for c, m in unknown_mask.items():
    df.loc[m, c] = pd.NA

# Numeric sentinels
df["rating_missing"] = raw["Rating"] == -1
df["Rating"] = raw["Rating"].replace(-1, np.nan)
df["founded_missing"] = raw["Founded"] == -1
df["Founded"] = raw["Founded"].replace(-1, np.nan)

for c in SENTINEL_COLS:
    log("sentinel -1 -> NaN", c, before[c], "sentinel encoding; MNAR by mechanism, not imputed")
for c, m in unknown_mask.items():
    log("sentinel 'Unknown*' -> NaN", c, m.sum(), "second sentinel system, independent of -1")
log("sentinel -1 -> NaN", "Rating", df["rating_missing"].sum(), "unrated company; flagged, not imputed")
log("sentinel -1 -> NaN", "Founded", df["founded_missing"].sum(), "founding year absent; flagged, not imputed")

miss = pd.DataFrame({
    "missing": df[SENTINEL_COLS + ["Rating", "Founded"]].isna().sum(),
    "pct": (100 * df[SENTINEL_COLS + ["Rating", "Founded"]].isna().mean()).round(1),
}).sort_values("missing", ascending=False)
print(miss.to_string())
print(f"\ntotal field values now correctly marked missing: {int(df.isna().sum().sum()):,}")

                   missing   pct
Competitors            501  74.6
Revenue                240  35.7
Founded                118  17.6
Industry                71  10.6
Sector                  71  10.6
Rating                  50   7.4
Size                    44   6.5
Headquarters            31   4.6
Type of ownership       31   4.6

total field values now correctly marked missing: 1,157


**So What:** `Competitors` (74.6%) is unusable as a primary variable and survives only as a
presence flag. `Revenue` (35.7%) and `Founded` (17.6%) are usable but every cut on them must state its
denominator — a "median company age" computed over the 82% that report one is not a statement about
the market, it is a statement about companies that disclose.

---

## 2.2 Salary parsing — creating the artifact, and labelling it as one

`salary_min` / `salary_max` / `salary_avg` are derived here because the forensic notebook needs them
and because the salary model (Stage 5b) is a first-class deliverable. **`batch_id` is derived
alongside them, from the same column**, which makes the dependency impossible to forget: these four
columns are all functions of one field, and that field indexes the scrape.

In [3]:
sal = df["Salary Estimate"].str.extract(r"\$(\d+)K-\$(\d+)K").astype(float)
df["salary_min"] = sal[0]
df["salary_max"] = sal[1]
df["salary_avg"] = (sal[0] + sal[1]) / 2
df["salary_range"] = sal[1] - sal[0]
df["salary_source"] = np.where(df["Salary Estimate"].str.contains("Employer"),
                               "Employer est.", "Glassdoor est.")

# The artifact, made explicit: batches numbered in order of first appearance.
first_seen = df.groupby("Salary Estimate")["index"].transform("min")
df["batch_id"] = first_seen.rank(method="dense").astype(int) - 1

log("parse to numeric", "salary_min/max/avg/range", len(df), "regex on '$NNK-$NNK'; units are $ thousands")
log("derive", "salary_source", int((df.salary_source == 'Employer est.').sum()),
    "20 rows are employer-supplied, a different provenance from the 652 Glassdoor estimates")
log("derive", "batch_id", len(df), "materialises the scrape-batch artifact found at Stage 1")

print(df[["salary_min", "salary_max", "salary_avg", "salary_range"]].describe().round(1).to_string())
print(f"\ndistinct salary_avg values: {df.salary_avg.nunique()}  across {len(df)} rows")
print(f"distinct batches:           {df.batch_id.nunique()}")
print(f"\nprovenance: {df.salary_source.value_counts().to_dict()}")

       salary_min  salary_max  salary_avg  salary_range
count       672.0       672.0       672.0         672.0
mean         99.2       148.1       123.7          48.9
std          33.0        48.0        39.6          23.0
min          31.0        56.0        43.5           4.0
25%          79.0       119.0       103.0          33.0
50%          91.0       133.0       114.0          52.0
75%         122.0       165.0       136.5          62.0
max         212.0       331.0       271.5         119.0

distinct salary_avg values: 26  across 672 rows
distinct batches:           30

provenance: {'Glassdoor est.': 652, 'Employer est.': 20}


## 2.3 Company, location and role parsing

In [4]:
# --- Company: strip the rating appended to the name in 622/672 rows -------------------
df["company_clean"] = df["Company Name"].str.split("\n").str[0].str.strip()
n_contaminated = int(raw["Company Name"].str.contains("\n").sum())
log("strip trailing rating", "company_clean", n_contaminated, "rating value appended to name string")

print(f"'Company Name' carried an appended rating in {n_contaminated}/{len(df)} rows.")
print(f"unique names before strip: {raw['Company Name'].nunique()}")
print(f"unique names after strip : {df['company_clean'].nunique()}")
print(f"collisions resolved      : {raw['Company Name'].nunique() - df['company_clean'].nunique()}")

'Company Name' carried an appended rating in 622/672 rows.
unique names before strip: 432
unique names after strip : 432
collisions resolved      : 0


> **A documented departure from `DOCS/DS_Jobs_Dataset_Analysis.md`.** The review lists "Company
> identity resolution" as a gap and suggests the same company appears under slightly different name
> strings. Stripping the rating suffix resolves **zero** collisions — 432 unique names before, 432
> after. Every company's rating is consistent across its own rows, so the suffix was redundant, not
> ambiguous. **No fuzzy name matching is performed:** it would risk false merges on 432 real company
> names to solve a problem that has not been demonstrated. If a genuine variant surfaces in EDA, this
> loops back — logged, per STRUCTURE.md's iteration rule.

In [5]:
# --- Location: normalise to a state code ---------------------------------------------
FULL_TO_ABBR = {"Utah": "UT", "New Jersey": "NJ", "Texas": "TX", "California": "CA"}
NON_STATE = {"United States", "Remote"}

def to_state(loc):
    if pd.isna(loc):
        return pd.NA
    tok = str(loc).split(",")[-1].strip()
    if tok in NON_STATE:
        return tok                      # kept as its own labelled category, NOT NaN
    return FULL_TO_ABBR.get(tok, tok)

df["job_state"] = df["Location"].map(to_state)
df["hq_state"] = df["Headquarters"].map(to_state)
df["is_remote"] = df["job_state"] == "Remote"
df["state_unspecified"] = df["job_state"].isin(["United States", "Remote"])

# same_state is NaN whenever either side is unknown or not a real state
valid = (~df["job_state"].isin(["United States", "Remote"]) & df["job_state"].notna()
         & ~df["hq_state"].isin(["United States", "Remote"]) & df["hq_state"].notna())
df["same_state"] = np.where(valid, df["job_state"] == df["hq_state"], np.nan)

log("normalise full state names -> codes", "job_state", 6, "Utah/New Jersey/Texas/California written in full")
log("keep as labelled category", "job_state", 17, "'United States' (11) and 'Remote' (6) are not states, not NaN")
log("derive", "same_state", int(valid.sum()), "NaN where either side unknown; never silently False")

print(f"distinct job states: {df['job_state'].nunique()}")
print(df["job_state"].value_counts().head(10).to_string())
print(f"\nsame_state computable for {int(valid.sum())}/{len(df)} rows; "
      f"of those, {df['same_state'].sum():.0f} are in-state ({100*df['same_state'].mean(skipna=True):.1f}%)")

distinct job states: 40
job_state
CA    166
VA     89
MA     62
NY     52
MD     40
IL     30
DC     26
TX     18
WA     16
OH     14

same_state computable for 624/672 rows; of those, 393 are in-state (63.0%)


In [6]:
# --- Role family and seniority from the title ----------------------------------------
title_l = df["Job Title"].str.lower()

def role_family(t):
    # ORDER MATTERS: 'machine learning' is tested first so that titles containing both
    # 'machine learning' and 'scientist' land in ML Engineer rather than Data Scientist.
    if re.search(r"machine learning|\bml\b|deep learning|ai engineer", t):
        return "ML Engineer"
    if re.search(r"data engineer|data engineering", t):
        return "Data Engineer"
    if re.search(r"analyst|analytics", t):
        return "Data Analyst"
    if re.search(r"data scientist|data science", t):
        return "Data Scientist"
    return "Other"

df["role_family"] = title_l.map(role_family)

SENIORITY = [("Director", r"director|vp|vice president|head of"),
             ("Manager",  r"manager|supervisor"),
             ("Principal", r"principal"),
             ("Lead",     r"\blead\b"),
             ("Staff",    r"\bstaff\b"),
             ("Senior",   r"senior|\bsr\.?\b")]

def seniority(t):
    for label, pat in SENIORITY:
        if re.search(pat, t):
            return label
    return "Unspecified"

df["seniority"] = title_l.map(seniority)
df["is_senior"] = df["seniority"] != "Unspecified"

log("derive", "role_family", len(df), "keyword map on Job Title; ML tested before DS to avoid mis-assignment")
log("derive", "seniority", int(df.is_senior.sum()), "keyword map; no Junior/Intern titles exist in this file")

print(df["role_family"].value_counts().to_string())
print()
print(df["seniority"].value_counts().to_string())
print(f"\nNOTE: 'Junior'/'Intern' match {int(title_l.str.contains(r'junior|intern').sum())} titles - "
      "the file has no entry level, which caps any seniority claim.")

role_family
Data Scientist    443
Data Analyst       72
ML Engineer        55
Other              55
Data Engineer      47

seniority
Unspecified    562
Senior          75
Principal        9
Manager          7
Lead             7
Staff            6
Director         6

NOTE: 'Junior'/'Intern' match 0 titles - the file has no entry level, which caps any seniority claim.


**So What:** the raw `Job Title` field has **172 distinct values and 337 rows (50.1%) reading
exactly "Data Scientist"** — it is simultaneously too granular to group on and too coarse to
distinguish roles. `role_family` fixes the first problem; the fact that seniority is *unspecified*
for 84% of postings caps what any seniority analysis can claim, and is reported rather than papered
over.

## 2.4 Company attributes → numeric

In [7]:
SIZE_MID = {"1 to 50 employees": 25, "51 to 200 employees": 125, "201 to 500 employees": 350,
            "501 to 1000 employees": 750, "1001 to 5000 employees": 3000,
            "5001 to 10000 employees": 7500, "10000+ employees": 15000}
REVENUE_MID = {
    "Less than $1 million (USD)": 0.5e6, "$1 to $5 million (USD)": 3e6,
    "$5 to $10 million (USD)": 7.5e6, "$10 to $25 million (USD)": 17.5e6,
    "$25 to $50 million (USD)": 37.5e6, "$50 to $100 million (USD)": 75e6,
    "$100 to $500 million (USD)": 300e6, "$500 million to $1 billion (USD)": 750e6,
    "$1 to $2 billion (USD)": 1.5e9, "$2 to $5 billion (USD)": 3.5e9,
    "$5 to $10 billion (USD)": 7.5e9, "$10+ billion (USD)": 15e9,
}
SCRAPE_YEAR = 2020   # ASSUMPTION - no date field exists; see limitations

df["size_mid"] = df["Size"].map(SIZE_MID)
df["revenue_mid"] = df["Revenue"].map(REVENUE_MID)
df["company_age"] = SCRAPE_YEAR - df["Founded"]

log("bucket -> midpoint", "size_mid", int(df.size_mid.notna().sum()),
    "'10000+' -> 15000 is an open-ended assumption, stated")
log("bucket -> midpoint", "revenue_mid", int(df.revenue_mid.notna().sum()),
    "'$10+ billion' -> $15bn is an open-ended assumption, stated")
log("derive", "company_age", int(df.company_age.notna().sum()),
    f"{SCRAPE_YEAR} - Founded; the scrape year is INFERRED, not given by the data")

unmapped_size = set(df.loc[df["Size"].notna() & df["size_mid"].isna(), "Size"])
unmapped_rev = set(df.loc[df["Revenue"].notna() & df["revenue_mid"].isna(), "Revenue"])
print(f"unmapped Size values   : {unmapped_size or 'none'}")
print(f"unmapped Revenue values: {unmapped_rev or 'none'}")
print(f"\ncompany_age: n={int(df.company_age.notna().sum())}, "
      f"median={df.company_age.median():.0f}y, max={df.company_age.max():.0f}y")

unmapped Size values   : none
unmapped Revenue values: none

company_age: n=554, median=21y, max=239y


## 2.5 Text-derived features

The description is the richest column in the file and the only one that carries structured
information about the *job* rather than the employer. Three families of feature come out of it.

The `R` flag deserves a note, because it is the one every notebook on this dataset gets wrong. A
naive `\br\b` on lowercased text fires on any standalone `r` — including inside accented words,
since `[a-z]` is ASCII-only and does not exclude `é`. The pattern used here is
**`(?<![^\W\d_])R(?![^\W\d_])` applied case-sensitively to the original text**: `[^\W\d_]` matches any
*Unicode* letter, so the lookarounds exclude accented neighbours, and requiring a capital `R` matches
how the language is actually written. It is validated against real match contexts below rather than
trusted.

In [8]:
desc_l = df["Job Description"].str.lower()

SKILLS = {
    "python": r"\bpython\b", "sql": r"\bsql\b",
    "spark": r"\bspark\b", "hadoop": r"\bhadoop\b", "hive": r"\bhive\b",
    "aws": r"\baws\b|amazon web services", "azure": r"\bazure\b",
    "gcp": r"\bgcp\b|google cloud", "tensorflow": r"tensorflow", "pytorch": r"pytorch|torch\b",
    "scikit": r"scikit|sklearn", "excel": r"\bexcel\b", "tableau": r"\btableau\b",
    "powerbi": r"power ?bi", "sas": r"\bsas\b", "scala": r"\bscala\b", "java": r"\bjava\b(?!script)",
    "docker": r"\bdocker\b", "kubernetes": r"\bkubernetes\b|\bk8s\b",
    "nosql": r"nosql|mongodb|cassandra", "git": r"\bgit\b|github",
}
for name, pat in SKILLS.items():
    df[f"skill_{name}"] = desc_l.str.contains(pat, regex=True)

# R is handled separately: case-SENSITIVE, on the ORIGINAL text, with Unicode-letter lookarounds.
R_PATTERN = r"(?<![^\W\d_])R(?![^\W\d_])"
df["skill_r"] = df["Job Description"].str.contains(R_PATTERN, regex=True)
SKILLS["r"] = R_PATTERN

DEGREES = {"phd": r"\bph\.?\s?d\b|doctorate", "masters": r"master'?s|\bm\.?s\.?\b|\bmsc\b",
           "bachelors": r"bachelor'?s|\bb\.?s\.?\b|\bbsc\b"}
for name, pat in DEGREES.items():
    df[f"degree_{name}"] = desc_l.str.contains(pat, regex=True)

df["desc_word_count"] = df["Job Description"].str.split().str.len()
df["desc_char_count"] = df["Job Description"].str.len()

# Years of experience: take the SMALLEST plausible match (the stated minimum).
def min_years(text):
    vals = [int(v) for v in re.findall(r"(\d{1,2})\s*\+?\s*(?:to\s*\d{1,2}\s*)?years?", text)]
    vals = [v for v in vals if 0 < v <= 25]      # discard parse noise like '2020 years'
    return min(vals) if vals else np.nan

df["years_experience_min"] = desc_l.map(min_years)

df["has_competitors"] = df["Competitors"].notna()
df["n_competitors"] = np.where(df["Competitors"].notna(),
                               df["Competitors"].fillna("").str.count(",") + 1, 0)

log("derive", "skill_* (22 flags)", len(df), "word-boundary regex on lowercased description")
log("derive", "degree_* (3 flags)", len(df), "credential requirement mentions")
log("derive", "years_experience_min", int(df.years_experience_min.notna().sum()),
    "min of plausible 'N years' matches; absent -> NaN, never 0")
log("derive", "desc_word_count", len(df), "posting detail proxy; 9-word minimum retained, not dropped")

skl = pd.DataFrame({"n": [df[f"skill_{k}"].sum() for k in SKILLS],
                    "pct": [round(100 * df[f'skill_{k}'].mean(), 1) for k in SKILLS]},
                   index=list(SKILLS)).sort_values("n", ascending=False)
print(skl.to_string())
print(f"\nyears_experience_min present in {int(df.years_experience_min.notna().sum())}/{len(df)} "
      f"({100*df.years_experience_min.notna().mean():.1f}%), median {df.years_experience_min.median():.0f}y")

              n   pct
python      489  72.8
sql         325  48.4
r           315  46.9
spark       179  26.6
aws         147  21.9
hadoop      143  21.3
java        140  20.8
tableau     123  18.3
tensorflow  103  15.3
nosql        95  14.1
scala        94  14.0
sas          82  12.2
scikit       80  11.9
excel        78  11.6
hive         70  10.4
git          67  10.0
pytorch      59   8.8
azure        50   7.4
powerbi      48   7.1
docker       38   5.7
gcp          33   4.9
kubernetes   15   2.2

years_experience_min present in 436/672 (64.9%), median 3y


In [9]:
# Validate the R pattern against real contexts rather than trusting the count.
print("Match contexts for skill_r:")
shown = 0
for txt in df.loc[df["skill_r"], "Job Description"]:
    m = re.search(R_PATTERN, txt)
    if m:
        ctx = txt[max(0, m.start() - 42):m.start() + 42].replace("\n", " ")
        print(f"   ...{ctx}...")
        shown += 1
    if shown == 5:
        break

naive = desc_l.str.contains(r"\br\b", regex=True)
print(f"\nskill_r fires on {int(df.skill_r.sum())} postings ({100*df.skill_r.mean():.1f}%)")
print(f"a naive lowercase \\br\\b would fire on {int(naive.sum())} - the two agree at the posting "
      "level here, but the strict pattern is the one that survives accented text.")

# Encoding check - does the corpus need repair before TF-IDF at Stage 5b?
n_nonascii = int(df["Job Description"].str.contains(r"[^\x00-\x7f]").sum())
n_replacement = int(df["Job Description"].str.contains("\ufffd").sum())
print(f"\nEncoding: {n_nonascii}/{len(df)} descriptions contain non-ASCII characters "
      f"(curly quotes, bullets, French accents);")
print(f"          {n_replacement} contain a U+FFFD replacement character.")
print("          -> the file is valid UTF-8. No mojibake repair is needed before tokenisation.")

Match contexts for skill_r:
   ...tals and data visualizations (e.g., using R/Shiny, Python/Flask, D3) Building and ma...
   ...ming languages; preferably Python, SAS or R · Familiar with one of the database tech...
   ...Programming experience (preferred Python, R, C++) preferably in Linux and high-perfo...
   ...ing algorithms using AWS. Experience with R, Java, or other programming languages. S...
   ... languages such as Python (preferred) and R. In this role, you will own and share te...

skill_r fires on 315 postings (46.9%)
a naive lowercase \br\b would fire on 314 - the two agree at the posting level here, but the strict pattern is the one that survives accented text.

Encoding: 343/672 descriptions contain non-ASCII characters (curly quotes, bullets, French accents);
          0 contain a U+FFFD replacement character.
          -> the file is valid UTF-8. No mojibake repair is needed before tokenisation.


## 2.6 Duplicates — three levels, three treatments

This is the most consequential cleaning decision in the project, so each level is counted and
justified separately.

In [10]:
# Level 1 - fully identical rows (ignoring the row id)
cols_nokey = [c for c in raw.columns if c != "index"]
exact_dup = raw.duplicated(cols_nokey, keep="first")

# Level 2 - the same posting text appearing under DIFFERENT salary bands
desc_groups = raw.groupby("Job Description")["Salary Estimate"].nunique()
repeated_desc = raw["Job Description"].value_counts()
multi_band = desc_groups[(repeated_desc > 1) & (desc_groups > 1)]

# Level 3 - posting identity, the CV grouping key
df["posting_id"] = (df.groupby(["company_clean", "Job Title", "Job Description"], dropna=False)
                      .ngroup())
df["is_exact_duplicate"] = exact_dup.values

print(f"L1  fully identical rows                 : {int(exact_dup.sum())}")
print(f"L2  descriptions appearing more than once: {int((repeated_desc > 1).sum())}")
print(f"      ...of which carry >1 salary band   : {len(multi_band)} "
      f"({100*len(multi_band)/int((repeated_desc > 1).sum()):.1f}%)")
print(f"L3  distinct posting_id values           : {df.posting_id.nunique()} from {len(df)} rows")

log("flag, do not drop", "is_exact_duplicate", int(exact_dup.sum()),
    "kept in the forensic base to preserve the 672-row batch block structure asserted at Stage 1")
log("preserve deliberately", "multi-band repeated postings", int(repeated_desc[repeated_desc > 1].sum()),
    "THESE ROWS ARE THE EVIDENCE for the central finding - dropping them would destroy it")
log("derive", "posting_id", df.posting_id.nunique(),
    "(company, title, description) identity; the GroupKFold key for every model")

L1  fully identical rows                 : 13
L2  descriptions appearing more than once: 151
      ...of which carry >1 salary band   : 149 (98.7%)
L3  distinct posting_id values           : 492 from 672 rows


### Why the middle level is kept

The instinct with 149 near-duplicate groups is to deduplicate and move on. Here that would delete the
finding. Those rows are the same posting — same company, same title, same description — appearing
under **different salary bands**, and their disagreement is the direct evidence that salary is not a
property of the job.

So the split is: **keep everything in the forensic base, deduplicate for market statistics.**

In [11]:
spread = (df[df.duplicated("Job Description", keep=False)]
          .groupby("Job Description")["salary_avg"].agg(["min", "max", "nunique"]))
spread["spread_k"] = spread["max"] - spread["min"]

print("Salary spread within a SINGLE identical posting ($K):")
print(spread["spread_k"].describe().round(1).to_string())
print(f"\npostings whose salary varies at all: "
      f"{int((spread.spread_k > 0).sum())}/{len(spread)} "
      f"({100*(spread.spread_k > 0).mean():.1f}%)")
print("\nWorst offenders:")
worst = spread.nlargest(3, "spread_k")
for d, row in worst.iterrows():
    r0 = df[df["Job Description"] == d].iloc[0]
    print(f"  {r0.company_clean[:34]:34s} | {r0['Job Title'][:40]:40s} "
          f"| ${row['min']:.0f}K -> ${row['max']:.0f}K  (spread ${row.spread_k:.0f}K)")

Salary spread within a SINGLE identical posting ($K):
count    151.0
mean      44.8
std       42.4
min        0.0
25%       16.0
50%       28.0
75%       61.5
max      179.0

postings whose salary varies at all: 149/151 (98.7%)

Worst offenders:
  Maxar Technologies                 | Senior Data Scientist                    | $92K -> $272K  (spread $179K)
  Southwest Research Institute       | ENGINEER - COMPUTER SCIENTIST - RESEARCH | $92K -> $272K  (spread $179K)
  Klaviyo                            | Machine Learning Engineer                | $92K -> $272K  (spread $179K)


## 2.7 Business-rule validation on the cleaned frame

In [12]:
checks = [
    ("row count preserved at 672", len(df) == 672),
    ("batch block structure still intact",
     bool((df.groupby("batch_id")["index"].agg(lambda s: s.max() - s.min() + 1)
           == df.groupby("batch_id").size()).all())),
    ("salary_min < salary_max everywhere", bool((df.salary_min < df.salary_max).all())),
    ("salary_avg between min and max",
     bool(df.salary_avg.between(df.salary_min, df.salary_max).all())),
    ("Rating in [1,5] or NaN", bool(df.Rating.dropna().between(1, 5).all())),
    ("company_age non-negative", bool((df.company_age.dropna() >= 0).all())),
    ("no -1 survives in any object column",
     not any((df[c].astype("string") == "-1").any() for c in df.columns if df[c].dtype != "float64")),
    ("role_family covers every row", bool(df.role_family.notna().all())),
    ("posting_id has no nulls", bool(df.posting_id.notna().all())),
    ("years_experience_min within (0,25]",
     bool(df.years_experience_min.dropna().between(1, 25).all())),
]
val = pd.DataFrame(checks, columns=["rule", "passed"])
val["result"] = np.where(val["passed"], "PASS", "FAIL")
print(val[["rule", "result"]].to_string(index=False))
assert val.passed.all(), "business-rule validation failed"
print(f"\n{val.passed.sum()}/{len(val)} business rules passed")

                               rule result
         row count preserved at 672   PASS
 batch block structure still intact   PASS
 salary_min < salary_max everywhere   PASS
     salary_avg between min and max   PASS
             Rating in [1,5] or NaN   PASS
           company_age non-negative   PASS
no -1 survives in any object column   PASS
       role_family covers every row   PASS
            posting_id has no nulls   PASS
 years_experience_min within (0,25]   PASS

10/10 business rules passed


## 2.8 Outputs

Two bases, and **which one an exhibit uses is stated on the exhibit**:

| File | Rows | Grain | Used for |
|---|---|---|---|
| `postings_clean.parquet` | 672 | posting-instance-in-batch | the forensic work (Stage 5a) and the salary model, which must see the file as delivered |
| `postings_dedup.parquet` | 489 | unique posting | **every market-structure statistic** — so a posting scraped five times does not vote five times |

In [13]:
dedup = df.sort_values("index").drop_duplicates("posting_id", keep="first").reset_index(drop=True)

skill_cols = [c for c in df.columns if c.startswith(("skill_", "degree_"))]
skills_matrix = dedup[["posting_id", "role_family", "seniority", "job_state"] + skill_cols].copy()

df.to_parquet(PROC / "postings_clean.parquet", index=False)
dedup.to_parquet(PROC / "postings_dedup.parquet", index=False)
skills_matrix.to_parquet(PROC / "skills_matrix.parquet", index=False)

cleaning_log = pd.DataFrame(LOG)
cleaning_log.to_csv(PROC / "_cleaning_log.csv", index=False)

print(f"postings_clean.parquet  {df.shape[0]:>4} x {df.shape[1]}")
print(f"postings_dedup.parquet  {dedup.shape[0]:>4} x {dedup.shape[1]}")
print(f"skills_matrix.parquet   {skills_matrix.shape[0]:>4} x {skills_matrix.shape[1]}")
print(f"_cleaning_log.csv       {len(cleaning_log):>4} actions logged")
print(f"\ndedup removed {len(df) - len(dedup)} re-observations "
      f"({100*(len(df)-len(dedup))/len(df):.1f}% of rows)")

postings_clean.parquet   672 x 67
postings_dedup.parquet   492 x 67
skills_matrix.parquet    492 x 29
_cleaning_log.csv         31 actions logged

dedup removed 180 re-observations (26.8% of rows)


In [14]:
print(cleaning_log.to_string(index=False, max_colwidth=62))

                             action                       column  rows_affected                                                            why
                 sentinel -1 -> NaN                 Headquarters             31              sentinel encoding; MNAR by mechanism, not imputed
                 sentinel -1 -> NaN                         Size             27              sentinel encoding; MNAR by mechanism, not imputed
                 sentinel -1 -> NaN            Type of ownership             27              sentinel encoding; MNAR by mechanism, not imputed
                 sentinel -1 -> NaN                     Industry             71              sentinel encoding; MNAR by mechanism, not imputed
                 sentinel -1 -> NaN                       Sector             71              sentinel encoding; MNAR by mechanism, not imputed
                 sentinel -1 -> NaN                      Revenue             27              sentinel encoding; MNAR by mechanism, not imputed

---

## Stage 2 — Gate Checklist

- [x] **Missing-value strategy documented per column** — two sentinel systems converted and counted;
      **nothing imputed**, with the MNAR reasoning stated (§2.1)
- [x] **Duplicates assessed and handled with justification** — three levels, three treatments; the
      middle level *deliberately preserved* as evidence (§2.6)
- [x] **All columns have correct dtypes** — enforced at ingestion, re-validated here
- [x] **Business-rule validations pass** — 10/10, including that the batch block structure survived
      cleaning intact
- [x] **Cleaning log exists** — every action with column, rows affected and reason
- [x] **Cleaned data saved to `data/processed/`** — two bases plus the skills matrix

**Loop-back log:** none from EDA yet. One departure from `DOCS/DS_Jobs_Dataset_Analysis.md` recorded
in §2.3 (company-name resolution resolves zero collisions; no fuzzy matching performed).

**Next:** `03_eda.ipynb` — hypothesis-driven exploration against the Stage 0 issue tree.